In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

In [58]:
df = pd.read_csv("../../Datasets/Processed/Dataset Dupilumab Baseline.xlsx - Anagrafica.csv")
df.shape

(112, 47)

In [59]:
df.head()

,Sesso,Età,Peso (kg),Altezza (cm),"Biologico precedente (0 no, 1 sì)",Categoria Professione,Comorbidità,Num Comorbidità,Familiarità Atopia,"Fumo Baseline (0=no, 1=ex, 2=si)",...,Ostruzione nasale,Rinorrea anteriore,Rinorrea posteriore,Dolore facciale,Starnuti,QoL,Cellule caliciformi mucipare,Citologico_Neutrofili,Citologico_Eosinofili,TARGET
0,F,61.0,66.0,167.0,0.0,Ufficio,asma,1,0.0,2,...,10.0,10.0,10.0,10.0,8.0,1.0,NaN,NaN,NaN,1
1,M,47.0,78.0,180.0,0.0,Ufficio,NaN,0,0.0,1,...,10.0,8.0,10.0,8.0,8.0,2.0,1.0,2.0,1.0,0
2,F,70.0,81.0,175.0,0.0,Disoccupati,"ipertensione, ipercolesterolemia, asma",3,1.0,1,...,8.0,6.0,8.0,0.0,8.0,3.0,1.0,1.0,2.0,1
3,F,36.0,56.0,159.0,1.0,Industria,asma,1,0.0,0,...,10.0,10.0,8.0,8.0,10.0,2.0,0.0,2.0,3.0,1
4,M,75.0,75.0,176.0,NaN,Disoccupati,"rinite, congiuntivite, k polmone, diabete 2, asma",5,0.0,1,...,8.0,4.0,8.0,4.0,8.0,5.0,NaN,NaN,NaN,0


In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 47 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Sesso                                            112 non-null    str    
 1   Età                                              110 non-null    float64
 2   Peso (kg)                                        102 non-null    float64
 3   Altezza (cm)                                     102 non-null    float64
 4   Biologico precedente (0 no, 1 sì)                50 non-null     float64
 5   Categoria Professione                            87 non-null     str    
 6   Comorbidità                                      71 non-null     str    
 7   Num Comorbidità                                  112 non-null    int64  
 8   Familiarità Atopia                               80 non-null     float64
 9   Fumo Baseline (0=no, 1=ex, 2=si)           

In [61]:
df.columns

Index(['Sesso', 'Età', 'Peso (kg)', 'Altezza (cm)',
       'Biologico precedente (0 no, 1 sì)', 'Categoria Professione',
       'Comorbidità', 'Num Comorbidità', 'Familiarità Atopia',
       'Fumo Baseline (0=no, 1=ex, 2=si)', 'Anni di astinenza',
       'Siga al giorno', 'Fumo Score', 'OCS nell'ultimo anno',
       'Numero di brevi cicli/anno di OCS',
       'Dosaggio cumulativo annuale in mg di prednisone',
       'Numero di pregressi interventi ESS', 'N. allergeni positivi',
       'Quali allergeni positivi', 'Categorie Allergeni',
       'Intolleranza ai FANS', 'Giorni al mese di utilizzo spray', 'ACT',
       'NPS', 'Neutrofili', 'Linfociti', 'Monociti', 'Eosinofili', 'Basofili',
       'IgE totali', 'Physical Symptoms', 'Health and QoL', 'Nasal Symptoms',
       'Otologic Symptoms', 'Sleep Symptoms', 'Emotional Symptoms', 'NCS',
       'Ostruzione nasale', 'Rinorrea anteriore', 'Rinorrea posteriore',
       'Dolore facciale', 'Starnuti', 'QoL', 'Cellule caliciformi mucipare',
   

In [62]:
y = df['TARGET']
df.drop(columns=['TARGET'], inplace=True)
df.shape, y.shape

C:\Users\letyl\AppData\Local\Temp\ipykernel_19664\381983694.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['TARGET'], inplace=True)


((112, 46), (112,))

In [63]:
continuous_cols = [
    "Peso (kg)",
    "Altezza (cm)",
    #"Fumo Score",
    #"Dosaggio cumulativo annuale in mg di prednisone",
    "Neutrofili",
    "Linfociti",
    "Monociti",
    "Eosinofili",
    "Basofili",
    "IgE totali"]

df[continuous_cols] = df[continuous_cols].apply(
    lambda col: col.astype("string")
                   .str.replace(".", "", regex=False)
                   .str.replace(",", ".", regex=False)
                   .astype("float64")
)

discrete_cols = [
    "Età",
    "Num Comorbidità",
    #"Anni di astinenza",
    #"Siga al giorno",
    #"Numero di brevi cicli/anno di OCS",
    "Numero di pregressi interventi ESS",
    "N. allergeni positivi",
    "Giorni al mese di utilizzo spray",
    "ACT",
    "NPS",
    "Physical Symptoms",
    "Health and QoL",
    "Nasal Symptoms",
    "Otologic Symptoms",
    "Sleep Symptoms",
    "Emotional Symptoms",
    "NCS",
    "Ostruzione nasale",
    "Rinorrea anteriore",
    "Rinorrea posteriore",
    "Dolore facciale",
    "Starnuti",
    "QoL"
]

df[discrete_cols] = df[discrete_cols].astype("float64")

categorical_cols = [
    "Categoria Professione",
    "Sesso",
    "Biologico precedente (0 no, 1 sì)",
    "Familiarità Atopia",
    "OCS nell'ultimo anno",
    "Intolleranza ai FANS"
]

multi_categorical_cols = [
    "Comorbidità",
    "Quali allergeni positivi",
    #"Categorie Allergeni"
]

ordinal_cols = [
    "Cellule caliciformi mucipare",
    "Citologico_Neutrofili",
    "Citologico_Eosinofili",
    "Fumo Baseline (0=no, 1=ex, 2=si)"
]

df[ordinal_cols] = df[ordinal_cols].astype("float64")

In [64]:
Counter(y) # balanced class target

Counter({0: 62, 1: 50})

In [65]:
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=42, train_size=0.8)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((89, 46), (23, 46), (89,), (23,))

In [67]:
ct = ColumnTransformer(
    [
        # Numerical continuous features: Missing values + Standardization
        ("num_cont", make_pipeline(SimpleImputer(strategy='mean'), StandardScaler()), continuous_cols),
         # Numerical discrete features: Missing values + Standardization
        ("num_discr", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), discrete_cols),
        # Categorical features: Missing values + One-Hot Encoding
        ("cat", make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='error')), categorical_cols),
        # Ordinal features
        ("ord", make_pipeline(SimpleImputer(strategy='most_frequent'), StandardScaler()), ordinal_cols)
    ],
    verbose_feature_names_out=False # =False --> will not prefix any feature names and will error if feature names are not unique.
)

In [68]:
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [69]:
# RANDOM FOREST CLASSIFIER
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
f1 = f1_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
print("RandomForestClassifier f1_score: ", f1)
print("RandomForestClassifier accuracy: ", acc)
print("RandomForestClassifier precision: ", precision)
print("RandomForestClassifier recall: ", recall)

RandomForestClassifier f1_score:  0.5436507936507937
RandomForestClassifier accuracy:  0.5652173913043478
RandomForestClassifier precision:  0.5714285714285714
RandomForestClassifier recall:  0.36363636363636365


In [70]:
# LINEAR SVC
svc = LinearSVC(max_iter=3000, random_state=42, class_weight='balanced', dual=True)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
f1 = f1_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
print("LinearSVC f1_score: ", f1)
print("LinearSVC accuracy: ", acc)
print("LinearSVC precision: ", precision)
print("LinearSVC recall: ", recall)

LinearSVC f1_score:  0.6057142857142856
LinearSVC accuracy:  0.6086956521739131
LinearSVC precision:  0.6
LinearSVC recall:  0.5454545454545454


In [71]:
# LOGISTIC REGRESSION
logr =  LogisticRegression(max_iter=3000, random_state=42, class_weight='balanced')
logr.fit(X_train, y_train)
y_pred = logr.predict(X_test)
f1 = f1_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
print("LogisticRegression f1_score: ", f1)
print("LogisticRegression accuracy: ", acc)
print("LogisticRegression precision: ", precision)
print("LogisticRegression recall: ", recall)


LogisticRegression f1_score:  0.5643939393939394
LogisticRegression accuracy:  0.5652173913043478
LogisticRegression precision:  0.5384615384615384
LogisticRegression recall:  0.6363636363636364


In [74]:
models = {
    'LinearSVC': {
        'model': LinearSVC(max_iter=3000, random_state=42, class_weight='balanced', dual=True),
        'params': {
            'C': [0.001, 0.01, 0.1, 1, 10], 
            'loss': ['hinge', 'squared_hinge']
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=3000, random_state=42, class_weight='balanced'),
        'params': {
            'C': [0.1, 1, 5],
            'solver': ['saga', 'lbfgs'],
            'l1_ratio': [0, 0.5, 1]
        }
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier(n_jobs=-1, random_state=42),
        'params': {
            'n_estimators': [50, 100, 500, 1000],
            'max_depth' : [None, 5, 10, 50, 100],
            'min_samples_leaf' : [1, 2, 5, 10],
            'max_features' : [None, 'sqrt', 'log2']
        }
    }

}

In [75]:
# GridSearchCV using 3 cross-validation folds
best_models = {}
for model_name, mp in models.items():
    print(f"Training {model_name}...")

    grid = GridSearchCV(
        estimator=mp['model'],
        param_grid=mp['params'],
        cv=3,
        scoring='f1_macro'
    )

    grid.fit(X_train, y_train)
    best_models[model_name] = grid.best_estimator_

    avg_f1 = grid.best_score_  

    print(f"{model_name} best params: {grid.best_params_}")
    print(f"{model_name} F1_macro: {avg_f1}\n")

Training LinearSVC...
LinearSVC best params: {'C': 1, 'loss': 'hinge'}
LinearSVC F1_macro: 0.646581759625238

Training LogisticRegression...


d:\ADSP\2026-P8-LINKS_ADSP-Dupilumab_Eosinophilia_Early_Prediction\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\ADSP\2026-P8-LINKS_ADSP-Dupilumab_Eosinophilia_Early_Prediction\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:493: FitFailedWarning: 
18 fits failed out of a total of 54.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "d:\ADSP\2026-P8-LINKS_ADSP-Dupilumab_Eosinophilia_Early_Prediction\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **f

LogisticRegression best params: {'C': 5, 'l1_ratio': 0.5, 'solver': 'saga'}
LogisticRegression F1_macro: 0.6650161030595814

Training RandomForestClassifier...
RandomForestClassifier best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 50}
RandomForestClassifier F1_macro: 0.6111111111111112



In [78]:
# Linear SVC
svc = LinearSVC(C=1, loss='hinge', max_iter=3000, random_state=42, class_weight='balanced', dual=True)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_test)
f1_macro_svc = f1_score(y_test, y_pred_svc, average='macro')
print(f"Macro F1 Score LinearSVC (Test set): {f1_macro_svc}")

Macro F1 Score LinearSVC (Test set): 0.6057142857142856


In [79]:
# Logistic Regression
logr = LogisticRegression(C =5, l1_ratio=0.5, solver='saga', max_iter=3000, random_state=42, class_weight='balanced')
logr.fit(X_train, y_train)
y_pred_logr = logr.predict(X_test)
f1_macro_logr = f1_score(y_test, y_pred_logr, average='macro')
print(f"Macro F1 Score LogisticRegression (Test set): {f1_macro_logr}")

Macro F1 Score LogisticRegression (Test set): 0.6057142857142856


In [80]:
# RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1, random_state=42, max_depth=None, max_features='sqrt', min_samples_leaf=2, n_estimators=50)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
f1_macro = f1_score(y_test, y_pred_rf, average='macro')
print(f"Macro F1 Score RandomForestClassifier (Test set): {f1_macro}")

Macro F1 Score RandomForestClassifier (Test set): 0.50682261208577
